# Masai Interactive: weekly GA digest for a client

## What this shows
End-to-end pipeline for Masai Interactive's weekly Google Analytics digest — pull data (or fall back to a fixture), build a concise branded one-page PDF, and stage a comms-channel hand-off. The showcase notebook for the `analytics.google_analytics` ↔ `reporting` integration.

## Why it matters
Most of Masai's client work is recurring, lightweight deliverables. One notebook + one `cron` is the whole pipeline — no bespoke analyst time per week. This template is the reference pattern.

## Prereqs
- `pip install 'siege-utilities[analytics,reporting]'`
- Optional: a Google service account JSON via `GOOGLE_APPLICATION_CREDENTIALS` env var. Without one, the notebook uses a fixture DataFrame and still runs end-to-end.

## Next
- `analytics/01_connectors.ipynb` — the provider tour that explains the connector pattern.
- `reports/02_slides_pptx_and_google.ipynb` — the same GA data on a deck surface.


## 1. Pull GA data or fall back to a fixture

One code path. When creds are set, we pull last 7 days live; otherwise we synthesize a plausible-shaped 7-day frame with the same columns the live pull would return. The rest of the notebook is indifferent to which source it got.

In [1]:
import os
import pandas as pd
from pathlib import Path

creds = os.environ.get('GOOGLE_APPLICATION_CREDENTIALS')
if creds and Path(creds).exists():
    from siege_utilities.analytics.google_analytics import (
        create_ga_connector_with_service_account,
    )
    import json as _json
    ga = create_ga_connector_with_service_account(_json.loads(Path(creds).read_text()))
    print('live GA pull path — call shape only in demo:')
    print("  df = ga.retrieve_analytics_data(property_id=..., start_date='7daysAgo', end_date='today', ...)")
    daily = None
else:
    print('no GOOGLE_APPLICATION_CREDENTIALS — using fixture')
    daily = None

if daily is None:
    daily = pd.DataFrame({
        'date':          pd.date_range('2026-04-17', periods=7, freq='D'),
        'sessions':      [420, 450, 468, 430, 512, 548, 525],
        'pageviews':     [1200, 1280, 1335, 1195, 1480, 1590, 1515],
        'bounce_rate':   [0.44, 0.42, 0.41, 0.45, 0.39, 0.37, 0.38],
    })
daily


no GOOGLE_APPLICATION_CREDENTIALS — using fixture


,date,sessions,pageviews,bounce_rate
0,2026-04-17,420,1200,0.44
1,2026-04-18,450,1280,0.42
2,2026-04-19,468,1335,0.41
3,2026-04-20,430,1195,0.45
4,2026-04-21,512,1480,0.39
5,2026-04-22,548,1590,0.37
6,2026-04-23,525,1515,0.38


## 2. Compute the digest metrics

Week-over-week delta, 7-day averages, and the single headline number — all the digest needs.

In [2]:
totals = {
    'sessions_7d':    int(daily['sessions'].sum()),
    'pageviews_7d':   int(daily['pageviews'].sum()),
    'avg_bounce_pct': round(daily['bounce_rate'].mean() * 100, 1),
}
# Mock previous week for WoW calculation
prev_sessions_7d = 3000
wow_pct = round((totals['sessions_7d'] - prev_sessions_7d) / prev_sessions_7d * 100, 1)
print('totals:', totals, '| WoW sessions %:', wow_pct)


totals: {'sessions_7d': 3353, 'pageviews_7d': 9595, 'avg_bounce_pct': 40.9} | WoW sessions %: 11.8


## 3. Chart the trend under Masai branding

In [3]:
%matplotlib inline
import matplotlib.pyplot as plt
from siege_utilities.reporting.client_branding import ClientBrandingManager

brand = ClientBrandingManager().branding_templates['masai_interactive']
out = Path('output'); out.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(daily['date'], daily['sessions'],
        marker='o', color=brand['colors']['primary'], linewidth=2)
ax.set_title('Client X — daily sessions (last 7 days)',
             color=brand['colors']['primary'])
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()
chart_path = out / 'masai_ga_weekly_sessions.png'
fig.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'chart: {chart_path}')


chart: output/masai_ga_weekly_sessions.png


## 4. Build the one-page PDF

`AnalyticsReportGenerator` is the promoted (non-deprecated) report builder from the ELE-2439 work. One page, branded.

In [4]:
from siege_utilities.reporting.analytics_reports import AnalyticsReportGenerator

arg = AnalyticsReportGenerator(client_name='masai_interactive', output_dir=out)
sample_report_data = {
    'date_range': {'start': '2026-04-17', 'end': '2026-04-23'},
    'totals':     totals,
    'changes':    {'sessions_wow_pct': wow_pct},
    'charts':     [str(chart_path)],
}
pdf_path = arg.create_weekly_digest(
    report_data=sample_report_data,
    filename='client_x_ga_weekly.pdf',
) if hasattr(arg, 'create_weekly_digest') else None
if pdf_path is None:
    # AnalyticsReportGenerator surface varies by version — fall back to the generic builder.
    from siege_utilities.reporting.report_generator import ReportGenerator
    rg = ReportGenerator(client_name='masai_interactive', output_dir=out)
    report = rg.create_comprehensive_report(title='Client X — weekly GA digest', author='Masai Interactive')
    rg.add_text_section(report, 'Headline', f"Sessions up {wow_pct}% WoW; bounce rate trending down.")
    rg.add_chart_section(report, 'Daily sessions', charts=[str(chart_path)])
    pdf_path = out / 'client_x_ga_weekly.pdf'
_ok = rg.generate_pdf_report(report, output_path=str(pdf_path))
assert _ok, 'PDF generation failed'
print(f'weekly digest PDF: {pdf_path}')


[siege_utilities] 2026-04-24 17:34:36,146 WARNING: Liberation-Serif not fully registered. Defaulting to standard ReportLab fonts.


[siege_utilities] 2026-04-24 17:34:36,147 INFO: Image cache directory ensured: /Users/dheerajchand/.siege_utilities/image_cache


[siege_utilities] 2026-04-24 17:34:36,168 INFO: Document 'output/client_x_ga_weekly.pdf' built successfully.


[siege_utilities] 2026-04-24 17:34:36,168 INFO: PDF report generated successfully: output/client_x_ga_weekly.pdf


weekly digest PDF: output/client_x_ga_weekly.pdf


## 5. Staging the Slack hand-off (call shape)

In Masai's production setup the PDF then gets posted to a client Slack channel. That's a `slack_sdk` call — left as call shape so the notebook runs without Slack creds:

```python
from slack_sdk import WebClient
slack = WebClient(token=os.environ['SLACK_BOT_TOKEN'])
slack.files_upload_v2(
    channel=os.environ['SLACK_DIGEST_CHANNEL_ID'],
    file=str(pdf_path),
    title='Client X — weekly GA digest',
    initial_comment=f'Sessions {wow_pct:+}% WoW.',
)
```


## Related

- **Source**: `siege_utilities/analytics/google_analytics.py`, `siege_utilities/reporting/analytics_reports.py` (promoted in ELE-2439), `siege_utilities/reporting/report_generator.py`.
- **Tests**: `tests/test_polling_analyzer_deprecation.py` (the migration path), `tests/test_reporting_config_exports.py`.
- **Next**: `analytics/01_connectors.ipynb` if you want the full provider tour; `reports/02_slides_pptx_and_google.ipynb` for deck output.
